In [ ]:
# =============
# 环境、路径与参数
# =============

import json
from pathlib import Path
from PIL import Image

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

CURRENT_DIR = Path(".").resolve()
PROJECT_ROOT = Path("..").resolve()

IMAGE_DIR = PROJECT_ROOT / "images"

MODEL_SAVE_DIR = CURRENT_DIR / "models"
INFERENCE_RESULT_DIR = CURRENT_DIR / "inference_results"
INFERENCE_RESULT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD_JSON = CURRENT_DIR / "test_results" / "threshold_search" / "best_threshold_auto.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GLOBAL_IMG_SIZE = 512
TILE_IMG_SIZE = 384

N_EVAL_TILES = 16
EVAL_TILE_SIZE = 512
EVAL_STRIDE = 256

FINETUNE_MODE = "layer4_last"

DROPOUT = 0.35
MAX_POOL_SCALE = 0.25

LOCAL_TOPK = 3
GLOBAL_WEIGHT = 0.15
LOCAL_WEIGHT = 0.85

MODEL_PREFIX = "fire_global_local_attn_convnext_tiny"

DEFAULT_THRESHOLD = 0.6

print("DEVICE:", DEVICE)
print("CURRENT_DIR:", CURRENT_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGE_DIR:", IMAGE_DIR)
print("MODEL_SAVE_DIR:", MODEL_SAVE_DIR)
print("INFERENCE_RESULT_DIR:", INFERENCE_RESULT_DIR)

In [ ]:
# =============
# 模型定义：与训练 notebook 保持一致
# =============

class ConvNeXtAttentionFireClassifier(nn.Module):
    def __init__(self, dropout=0.35, use_pretrained=False, max_pool_scale=0.25):
        super().__init__()

        if use_pretrained:
            convnext = models.convnext_tiny(
                weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
            )
        else:
            convnext = models.convnext_tiny(weights=None)

        self.backbone = convnext.features
        self.feature_dim = 768
        self.max_pool_scale = 1.0 if max_pool_scale is None else float(max_pool_scale)

        self.attention = nn.Sequential(
            nn.Conv2d(self.feature_dim, 256, kernel_size=1),
            nn.GELU(),
            nn.Dropout2d(p=0.10),
            nn.Conv2d(256, 1, kernel_size=1)
        )

        fusion_dim = self.feature_dim * 3

        self.fc = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Dropout(p=dropout),

            nn.Linear(fusion_dim, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(p=0.35),

            nn.Linear(512, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(p=0.25),

            nn.Linear(128, 1)
        )

    def forward_features(self, x):
        return self.backbone(x)

    def get_attention_map(self, feature_map):
        attn_logits = self.attention(feature_map)
        B, _, H, W = attn_logits.shape
        attention_map = torch.softmax(
            attn_logits.flatten(2),
            dim=-1
        ).reshape(B, 1, H, W)
        return attention_map

    def attention_pooling(self, feature_map):
        B, C, H, W = feature_map.shape
        attn_logits = self.attention(feature_map).flatten(1)
        attn_weights = torch.softmax(attn_logits, dim=1).unsqueeze(-1)
        tokens = feature_map.flatten(2).transpose(1, 2)
        return (tokens * attn_weights).sum(dim=1)

    def forward(self, x, return_attention=False):
        feature_map = self.forward_features(x)

        attn_feature = self.attention_pooling(feature_map)
        avg_feature = F.adaptive_avg_pool2d(feature_map, output_size=1).flatten(1)

        max_feature = F.adaptive_max_pool2d(feature_map, output_size=1).flatten(1)
        max_feature = max_feature * self.max_pool_scale

        fused_feature = torch.cat([attn_feature, avg_feature, max_feature], dim=1)
        logit = self.fc(fused_feature)

        if return_attention:
            return logit, self.get_attention_map(feature_map)

        return logit


def set_finetune_mode(model, finetune_mode="layer4_last"):
    for param in model.parameters():
        param.requires_grad = False

    for param in model.attention.parameters():
        param.requires_grad = True

    for param in model.fc.parameters():
        param.requires_grad = True

    if finetune_mode == "head":
        pass

    elif finetune_mode == "layer4_last":
        for param in model.backbone[7][-1].parameters():
            param.requires_grad = True

    elif finetune_mode == "layer4_all":
        for param in model.backbone[7].parameters():
            param.requires_grad = True

    elif finetune_mode == "all":
        for param in model.parameters():
            param.requires_grad = True

    else:
        raise ValueError(f"Unknown finetune_mode: {finetune_mode}")

    return model


class GlobalLocalFireClassifier(nn.Module):
    def __init__(
        self,
        dropout=0.35,
        use_pretrained=False,
        max_pool_scale=0.25,
        topk=3,
        global_weight=0.15,
        local_weight=0.85
    ):
        super().__init__()

        self.topk = int(topk)
        self.global_weight = float(global_weight)
        self.local_weight = float(local_weight)

        self.shared_classifier = ConvNeXtAttentionFireClassifier(
            dropout=dropout,
            use_pretrained=use_pretrained,
            max_pool_scale=max_pool_scale
        )

    def _masked_topk_mean(self, tile_logits, tile_mask):
        B, N = tile_logits.shape
        local_logits = []
        topk_indices = []

        for b in range(B):
            valid = tile_mask[b].bool()
            vals = tile_logits[b][valid]

            if vals.numel() == 0:
                vals = tile_logits[b][:1]
                valid_indices = torch.arange(1, device=tile_logits.device)
            else:
                valid_indices = torch.where(valid)[0]

            k = min(self.topk, vals.numel())
            top = torch.topk(vals, k=k, largest=True)

            local_logits.append(top.values.mean())
            topk_indices.append(valid_indices[top.indices])

        return torch.stack(local_logits, dim=0), topk_indices

    def forward(self, global_image, tiles, tile_mask=None):
        B, N, C, H, W = tiles.shape

        if tile_mask is None:
            tile_mask = torch.ones(B, N, dtype=torch.bool, device=tiles.device)
        else:
            tile_mask = tile_mask.bool()

        global_logit, global_attention = self.shared_classifier(
            global_image,
            return_attention=True
        )
        global_logit = global_logit.squeeze(1)

        flat_tiles = tiles.reshape(B * N, C, H, W)
        flat_tile_logits, flat_tile_attn = self.shared_classifier(
            flat_tiles,
            return_attention=True
        )

        tile_logits = flat_tile_logits.squeeze(1).reshape(B, N)
        _, _, Ah, Aw = flat_tile_attn.shape
        tile_attentions = flat_tile_attn.reshape(B, N, 1, Ah, Aw)

        local_logit, topk_indices = self._masked_topk_mean(tile_logits, tile_mask)

        final_logit = (
            self.global_weight * global_logit +
            self.local_weight * local_logit
        )

        return {
            "logit": final_logit.unsqueeze(1),
            "global_logit": global_logit.unsqueeze(1),
            "local_logit": local_logit.unsqueeze(1),
            "tile_logits": tile_logits,
            "global_attention": global_attention,
            "tile_attentions": tile_attentions,
            "topk_indices": topk_indices
        }


def build_global_local_fire_model(
    finetune_mode="layer4_last",
    use_pretrained=False,
    dropout=0.35,
    max_pool_scale=0.25,
    topk=3,
    global_weight=0.15,
    local_weight=0.85
):
    model = GlobalLocalFireClassifier(
        dropout=dropout,
        use_pretrained=use_pretrained,
        max_pool_scale=max_pool_scale,
        topk=topk,
        global_weight=global_weight,
        local_weight=local_weight
    )

    model.shared_classifier = set_finetune_mode(
        model.shared_classifier,
        finetune_mode=finetune_mode
    )

    return model

In [ ]:
# =============
# Transform、滑窗 tile 与模型加载
# =============

global_eval_transform = transforms.Compose([
    transforms.Resize((GLOBAL_IMG_SIZE, GLOBAL_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

tile_eval_transform = transforms.Compose([
    transforms.Resize((TILE_IMG_SIZE, TILE_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def safe_torch_load(path, map_location="cpu", weights_only=False):
    try:
        return torch.load(path, map_location=map_location, weights_only=weights_only)
    except TypeError:
        return torch.load(path, map_location=map_location)


def make_eval_grid_boxes(image_w, image_h, tile_size=512, stride=256):
    tile_w = min(int(tile_size), int(image_w))
    tile_h = min(int(tile_size), int(image_h))

    if image_w <= tile_w:
        xs = [0]
    else:
        xs = list(range(0, image_w - tile_w + 1, stride))
        if xs[-1] != image_w - tile_w:
            xs.append(image_w - tile_w)

    if image_h <= tile_h:
        ys = [0]
    else:
        ys = list(range(0, image_h - tile_h + 1, stride))
        if ys[-1] != image_h - tile_h:
            ys.append(image_h - tile_h)

    return [(x, y, x + tile_w, y + tile_h) for y in ys for x in xs]


def make_tiles_for_image(raw_image):
    image_w, image_h = raw_image.size

    boxes = make_eval_grid_boxes(
        image_w=image_w,
        image_h=image_h,
        tile_size=EVAL_TILE_SIZE,
        stride=EVAL_STRIDE
    )

    if len(boxes) > N_EVAL_TILES:
        idxs = np.linspace(0, len(boxes) - 1, N_EVAL_TILES).astype(int)
        boxes = [boxes[i] for i in idxs]

    tile_tensors = []
    tile_mask = []

    for box in boxes:
        tile_img = raw_image.crop(box)
        tile_tensors.append(tile_eval_transform(tile_img))
        tile_mask.append(True)

    while len(tile_tensors) < N_EVAL_TILES:
        tile_tensors.append(tile_eval_transform(raw_image))
        tile_mask.append(False)

    tiles = torch.stack(tile_tensors, dim=0)
    tile_mask = torch.tensor(tile_mask, dtype=torch.bool)

    return tiles, tile_mask, boxes


def load_threshold(default_threshold=DEFAULT_THRESHOLD):
    if THRESHOLD_JSON.exists():
        with open(THRESHOLD_JSON, "r", encoding="utf-8") as f:
            info = json.load(f)

        threshold = float(info["best_threshold"])
        print("已读取自动搜索 threshold:", threshold)
        print("threshold 文件:", THRESHOLD_JSON)
        return threshold

    print("未找到自动 threshold 文件，使用默认 threshold:", default_threshold)
    return float(default_threshold)


def discover_model_paths(checkpoint_type="best"):
    pattern = f"{MODEL_PREFIX}_*_{checkpoint_type}.pth"
    model_paths = sorted(MODEL_SAVE_DIR.glob(pattern))

    if len(model_paths) == 0:
        # 兼容用户旧命名或不同 checkpoint 类型
        model_paths = sorted(MODEL_SAVE_DIR.glob(f"*_{checkpoint_type}.pth"))

    if len(model_paths) == 0:
        raise FileNotFoundError(f"未找到模型权重。搜索路径: {MODEL_SAVE_DIR}")

    return model_paths


def load_models(checkpoint_type="best"):
    model_paths = discover_model_paths(checkpoint_type=checkpoint_type)
    models_list = []

    for model_path in model_paths:
        checkpoint = safe_torch_load(
            model_path,
            map_location=DEVICE,
            weights_only=False
        )

        global_img_size = int(checkpoint.get("global_img_size", GLOBAL_IMG_SIZE))
        tile_img_size = int(checkpoint.get("tile_img_size", TILE_IMG_SIZE))

        if global_img_size != GLOBAL_IMG_SIZE or tile_img_size != TILE_IMG_SIZE:
            print(
                "警告：checkpoint 尺寸与当前推理参数不一致：",
                "ckpt global/tile=", global_img_size, tile_img_size,
                "current global/tile=", GLOBAL_IMG_SIZE, TILE_IMG_SIZE
            )

        model = build_global_local_fire_model(
            finetune_mode=checkpoint.get("finetune_mode", FINETUNE_MODE),
            use_pretrained=False,
            dropout=float(checkpoint.get("dropout", DROPOUT)),
            max_pool_scale=float(checkpoint.get("max_pool_scale", MAX_POOL_SCALE)),
            topk=int(checkpoint.get("local_topk", LOCAL_TOPK)),
            global_weight=float(checkpoint.get("global_weight", GLOBAL_WEIGHT)),
            local_weight=float(checkpoint.get("local_weight", LOCAL_WEIGHT))
        )

        state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
        model.load_state_dict(state_dict, strict=True)
        model.to(DEVICE)
        model.eval()

        models_list.append(model)
        print("Loaded:", model_path)

    return models_list, model_paths


def resolve_image_path(input_image_path):
    p = Path(str(input_image_path)).expanduser()

    if p.exists():
        return p

    p2 = PROJECT_ROOT / p
    if p2.exists():
        return p2

    p3 = IMAGE_DIR / p.name
    if p3.exists():
        return p3

    raise FileNotFoundError(f"输入图片不存在: {input_image_path}")

In [ ]:
# =============
# 单图推理函数
# =============

@torch.no_grad()
def predict_one_image(image_path, models_list, threshold):
    image_path = resolve_image_path(image_path)

    raw_image = Image.open(image_path).convert("RGB")

    global_tensor = global_eval_transform(raw_image).unsqueeze(0).to(DEVICE)
    tiles, tile_mask, tile_boxes = make_tiles_for_image(raw_image)

    tiles = tiles.unsqueeze(0).to(DEVICE)
    tile_mask = tile_mask.unsqueeze(0).to(DEVICE)

    probs = []
    global_probs = []
    local_probs = []
    tile_probs_all = []
    topk_boxes_all = []

    for model in models_list:
        outputs = model(
            global_image=global_tensor,
            tiles=tiles,
            tile_mask=tile_mask
        )

        logit = outputs["logit"].squeeze()
        global_logit = outputs["global_logit"].squeeze()
        local_logit = outputs["local_logit"].squeeze()
        tile_logits = outputs["tile_logits"].squeeze(0)

        prob = torch.sigmoid(logit).item()
        global_prob = torch.sigmoid(global_logit).item()
        local_prob = torch.sigmoid(local_logit).item()
        tile_probs = torch.sigmoid(tile_logits).detach().cpu().numpy()

        probs.append(prob)
        global_probs.append(global_prob)
        local_probs.append(local_prob)
        tile_probs_all.append(tile_probs.tolist())

        valid_tile_probs = [
            (i, float(tile_probs[i]))
            for i in range(len(tile_boxes))
        ]
        valid_tile_probs = sorted(valid_tile_probs, key=lambda x: x[1], reverse=True)
        topk = valid_tile_probs[:LOCAL_TOPK]

        topk_boxes = []
        for i, p in topk:
            box = tile_boxes[i]
            topk_boxes.append({
                "tile_index": int(i),
                "tile_prob_fire": float(p),
                "box_xyxy": [int(v) for v in box]
            })

        topk_boxes_all.append(topk_boxes)

    probs = np.array(probs, dtype=np.float32)
    global_probs = np.array(global_probs, dtype=np.float32)
    local_probs = np.array(local_probs, dtype=np.float32)

    mean_prob = float(probs.mean())
    mean_global_prob = float(global_probs.mean())
    mean_local_prob = float(local_probs.mean())
    pred = int(mean_prob >= threshold)

    result = {
        "image_path": str(image_path),
        "filename": image_path.name,
        "prob_fire": mean_prob,
        "prob_global": mean_global_prob,
        "prob_local": mean_local_prob,
        "threshold": float(threshold),
        "pred": pred,
        "result": "fire" if pred == 1 else "no_fire",
        "tile_boxes": [
            [int(v) for v in box]
            for box in tile_boxes
        ],
        "topk_tile_boxes_by_model": topk_boxes_all
    }

    for i, prob in enumerate(probs):
        result[f"model_{i:02d}_prob_fire"] = float(prob)
        result[f"model_{i:02d}_prob_global"] = float(global_probs[i])
        result[f"model_{i:02d}_prob_local"] = float(local_probs[i])
        result[f"model_{i:02d}_tile_probs"] = tile_probs_all[i]

    return result

In [ ]:
# =============
# 输入任意一张图片地址，判断图片是否有火
# =============

INPUT_IMAGE_PATH = r"../images/20250526_firesmoke_01294.jpg"

CHECKPOINT_TYPE = "best"

THRESHOLD = load_threshold(default_threshold=DEFAULT_THRESHOLD)

models_list, model_paths = load_models(checkpoint_type=CHECKPOINT_TYPE)

result = predict_one_image(
    image_path=INPUT_IMAGE_PATH,
    models_list=models_list,
    threshold=THRESHOLD
)

result_df = pd.DataFrame([{
    k: v for k, v in result.items()
    if not isinstance(v, (list, dict))
}])

save_csv = INFERENCE_RESULT_DIR / "single_image_prediction.csv"
save_json = INFERENCE_RESULT_DIR / "single_image_prediction.json"

result_df.to_csv(save_csv, index=False, encoding="utf-8-sig")

with open(save_json, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\n输入图片:")
print(result["image_path"])

print("\n每个模型概率:")
for key, value in result.items():
    if key.startswith("model_") and (
        key.endswith("_prob_fire") or key.endswith("_prob_global") or key.endswith("_prob_local")
    ):
        print(key, ":", round(value, 6))

print("\n集成结果:")
print("prob_fire:", round(result["prob_fire"], 6))
print("prob_global:", round(result["prob_global"], 6))
print("prob_local:", round(result["prob_local"], 6))
print("threshold:", result["threshold"])
print("pred:", result["pred"])
print("result:", result["result"])

print("\nTop-k tile 区域:")
print(json.dumps(result["topk_tile_boxes_by_model"], ensure_ascii=False, indent=2))

print("\n推理结果已保存:")
print(save_csv)
print(save_json)

In [ ]:
# =============
# 可选：批量推理一个文件夹
# =============

# INPUT_DIR = PROJECT_ROOT / "images"
# MAX_IMAGES = 50
#
# image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
# image_paths = [
#     p for p in INPUT_DIR.rglob("*")
#     if p.is_file() and p.suffix.lower() in image_exts
# ][:MAX_IMAGES]
#
# batch_rows = []
#
# for p in image_paths:
#     r = predict_one_image(
#         image_path=p,
#         models_list=models_list,
#         threshold=THRESHOLD
#     )
#     batch_rows.append({
#         k: v for k, v in r.items()
#         if not isinstance(v, (list, dict))
#     })
#
# batch_df = pd.DataFrame(batch_rows)
# batch_csv = INFERENCE_RESULT_DIR / "batch_image_predictions.csv"
# batch_df.to_csv(batch_csv, index=False, encoding="utf-8-sig")
#
# print("批量推理完成:")
# print(batch_csv)